# Taxi Anomaly Detector - Environment Check

Run this notebook first to verify MinIO connectivity and that NYC TLC parquet data was imported during chart installation.

Upstream taxi data can be sourced from **Zetaris** (installed in the cluster); this quickstart stores working copies in MinIO buckets `dev-data` and `data`.

In [ ]:
import os

print("MinIO endpoint:", os.getenv("AWS_S3_ENDPOINT"))
print("Default bucket:", os.getenv("AWS_S3_BUCKET"))
print("Dev bucket:", os.getenv("TAXI_DEV_DATA_BUCKET", "dev-data"))
print("Data prefix:", os.getenv("TAXI_DATA_PREFIX", "tlc-trip-data"))
print("Trip type:", os.getenv("TAXI_DATA_TRIP_TYPE", "yellow_tripdata"))

In [ ]:
import boto3
import pandas as pd

endpoint = os.environ["AWS_S3_ENDPOINT"]
access_key = os.environ["AWS_ACCESS_KEY_ID"]
secret_key = os.environ["AWS_SECRET_ACCESS_KEY"]
data_bucket = os.getenv("TAXI_DATA_BUCKET", os.getenv("AWS_S3_BUCKET", "data"))
dev_bucket = os.getenv("TAXI_DEV_DATA_BUCKET", "dev-data")
prefix = os.getenv("TAXI_DATA_PREFIX", "tlc-trip-data")
trip_type = os.getenv("TAXI_DATA_TRIP_TYPE", "yellow_tripdata")

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

# --- Check dev-data bucket (informational only) ---
print(f"Checking s3://{dev_bucket}/")
dev_objects = s3.list_objects_v2(Bucket=dev_bucket).get("Contents", [])
if dev_objects:
    print("Objects:", [obj["Key"] for obj in dev_objects])
else:
    print(f"Bucket '{dev_bucket}' is empty (this is expected before development begins).")

# --- Check data bucket: list parquet files under prefix, filter by trip type ---
print(f"\nChecking s3://{data_bucket}/{prefix}/")
paginator = s3.get_paginator("list_objects_v2")
all_keys = []
for page in paginator.paginate(Bucket=data_bucket, Prefix=prefix):
    all_keys.extend([obj["Key"] for obj in page.get("Contents", [])])

parquet_files = sorted([k for k in all_keys if k.endswith(".parquet") and trip_type in k])
print(f"Found {len(parquet_files)} parquet file(s) matching trip type '{trip_type}':")
for f in parquet_files:
    print(f"  {f}")

if not parquet_files:
    raise RuntimeError(
        f"No parquet files matching '{trip_type}' found under s3://{data_bucket}/{prefix}/. "
        "Has the data-import job completed?"
    )

# --- Load a sample from the first parquet file ---
sample_key = parquet_files[0]
print(f"\nLoading sample from s3://{data_bucket}/{sample_key} ...")
obj = s3.get_object(Bucket=data_bucket, Key=sample_key)
df = pd.read_parquet(obj["Body"])
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
display(df.head())

print("\nAll checks passed.")